python scripts/reformat_einstein.py --datadir data --rename data/rename_columns.xlsx --correction data/fix_values.xlsx --output combined_test_einstein.tsv

In [1]:
import pandas as pd

In [2]:
df_combined = pd.read_csv('../combined_test_einstein.tsv', sep='\t')

In [3]:
df_combined['state'].unique()

array(['São Paulo', 'RIO DE JANEIRO', 'Goiás'], dtype=object)

In [4]:
df_combined['test_kit'].unique()

array(['covid_pcr', 'covid_antigen', 'test_3', 'test_2', 'test_4',
       'vsr_antigen'], dtype=object)

In [5]:
df_dict = pd.read_excel(
    './../data/EINSTEIN/20230814_Dados_Epidemiologicos_Semanais_ITpS_SE32.xlsx', 
    index_col=None, header=0, 
    sheet_name=None, 
    dtype='str'
)

dfs_paineis = []
for sheet in df_dict.keys():
    if sheet.startswith('EXAME'):
        # einstein
        continue    
    dfs_paineis.append(df_dict[sheet].assign(ExcelSheet=sheet))

df_paineis = pd.concat(dfs_paineis, ignore_index=True)

In [6]:
df_paineis['ACCESSION'] = df_paineis['ACCESSION'].astype(int)

## Duplicatas

Duplicates no sample_id

In [7]:
(
    df_combined[['sample_id']]
    .assign( duplicated = df_combined.duplicated(subset=['sample_id']))
    .query('duplicated == True')
)

,sample_id,duplicated


Duplicates no test_id

In [8]:
df_duplicates = (
    df_combined
    # .query("test_kit == 'unknown'")
    .assign( one=1 )
    .groupby(['test_id'])
    .agg(num_duplicates=('one', 'sum'))
    .query("num_duplicates > 1")
    .assign(num_duplicates=lambda x: x['num_duplicates']-1)
)


print( df_duplicates['num_duplicates'].sum(), df_duplicates['num_duplicates'].max(), df_duplicates['num_duplicates'].min() )
df_duplicates.sort_values('num_duplicates', ascending=False).head(10)

241454 7 1


,num_duplicates
test_id,
2022082011929,7
2022102013848,7
2022274007730,7
2022243009341,7
2022054009352,7
2022332004751,7
2022017006199,7
2022127002379,7
2022013011347,7


In [9]:
df_multiple_tests = (
    df_combined
    .groupby('test_id')
    .agg(kits=('test_kit',lambda x: ','.join(x.tolist())))
    .query("kits.str.contains(',')", engine='python') 
)

In [10]:
df_multiple_tests.head(10)

,kits
test_id,
2022001000793,"test_3,test_3,test_3"
2022001000846,"test_2,test_2"
2022001000861,"test_2,test_2"
2022001000869,"test_2,test_2"
2022001000901,"test_2,test_2"
2022001000930,"test_2,test_2"
2022001001221,"test_2,test_2"
2022001001238,"test_2,test_2"
2022001001264,"test_2,test_2"


In [11]:
df_multiple_tests['kits'].unique()

array(['test_3,test_3,test_3', 'test_2,test_2', 'covid_pcr,test_2,test_2',
       'test_2,test_2,covid_antigen', 'covid_pcr,covid_antigen',
       'covid_pcr,test_2,test_2,covid_antigen',
       'test_4,test_4,test_4,test_4,test_4', 'covid_pcr,covid_pcr',
       'covid_pcr,test_2', 'test_2,test_2,covid_pcr',
       'covid_pcr,covid_pcr,test_2,test_2',
       'covid_pcr,test_4,test_4,test_4,test_4,test_4',
       'test_4,test_4,test_4,test_4,test_4,covid_antigen',
       'covid_pcr,test_4,test_4,test_4,test_4,test_4,test_2,test_2',
       'test_4,test_4,test_4,test_4,test_4,test_2,test_2,covid_antigen',
       'test_4,test_4,test_4,test_4,test_4,covid_pcr',
       'test_4,test_4,test_4,test_4,test_4,test_2,test_2',
       'test_4,test_4,test_4,test_4,covid_pcr',
       'test_4,test_4,test_4,test_4', 'covid_pcr,vsr_antigen',
       'vsr_antigen,test_2,test_2',
       'covid_pcr,test_4,test_4,test_4,test_4',
       'test_3,test_3,test_3,test_4,test_4,test_4,test_4,test_4',
       'covid_a

In [12]:
df_combined.columns

Index(['lab_id', 'test_id', 'test_kit', 'patient_id', 'sample_id', 'state',
       'location', 'date_testing', 'epiweek', 'age', 'sex', 'FLUA_test_result',
       'Ct_FluA', 'FLUB_test_result', 'Ct_FluB', 'VSR_test_result', 'Ct_VSR',
       'SC2_test_result', 'Ct_geneE', 'Ct_geneN', 'Ct_geneS', 'Ct_ORF1ab',
       'Ct_RDRP', 'geneS_detection', 'META_test_result', 'RINO_test_result',
       'PARA_test_result', 'ADENO_test_result', 'BOCA_test_result',
       'COVS_test_result', 'ENTERO_test_result', 'BAC_test_result'],
      dtype='object')

## Uniqueness

In [13]:
if 'unknown' in df_combined['test_kit'].unique():
    print('WARNING! unknown test kits found')

df_combined['test_kit'].unique()

array(['covid_pcr', 'covid_antigen', 'test_3', 'test_2', 'test_4',
       'vsr_antigen'], dtype=object)

In [14]:
df_combined['state'].unique()

array(['São Paulo', 'RIO DE JANEIRO', 'Goiás'], dtype=object)

In [15]:
df_combined['location'].unique()

array(['GUARULHOS', 'SÃO PAULO', 'RIO DE JANEIRO', nan, 'SOROCABA',
       'GOIÂNIA'], dtype=object)

In [16]:
df_combined['date_testing'].max(), df_combined['date_testing'].min()

('2023-08-14', '2022-01-01')

In [17]:
df_combined['age'].unique(), df_combined['age'].min(), df_combined['age'].max()

(array([ 61,  53,  58,  49,  17,  56,  12,  16,  45,  26,  54,  14,  23,
         36,  48,  51,  50,  82,  59,  15,  55,  22,  35,  27,  60,  46,
          1,   3,  24,  37,  28,  42,  20,  41,  10,   8,  21,  40,   4,
         39,  34,  33,   6,  30,  87,  43,   7,  32,  38,  67,  74,  25,
         79,  65,  19,  31,  29,  11,   9,  62,  44,   2,  47,  57,  18,
         52,  84,  64,   5,  70,  75,  63,  71,  72,  77,  13,  86,  69,
         73,  68,  66,  93,  83,  80,  85,  76,  89,  94,  95,  81,  96,
         97,  88,  78,  91, 101, 100,  90,  92,  99,  98, 103, 102, 123,
        110, 104, 107, 108, 105,   0, 106, 112]),
 0,
 123)

In [18]:
df_combined['sex'].unique()

array(['M', 'F', 'I'], dtype=object)

In [19]:
for column in df_combined.columns:
    if column.endswith('_result'):
        print(column, df_combined[column].unique())

FLUA_test_result ['NT' 'Neg' 'Pos']
FLUB_test_result ['NT' 'Neg' 'Pos']
VSR_test_result ['NT' 'Neg' 'Pos']
SC2_test_result ['Neg' 'Pos' 'NT']
META_test_result ['NT']
RINO_test_result ['NT']
PARA_test_result ['NT']
ADENO_test_result ['NT']
BOCA_test_result ['NT']
COVS_test_result ['NT']
ENTERO_test_result ['NT']
BAC_test_result ['NT']


In [20]:
for col in df_combined.columns:
    if col.endswith('_result'):
        print(col)

FLUA_test_result
FLUB_test_result
VSR_test_result
SC2_test_result
META_test_result
RINO_test_result
PARA_test_result
ADENO_test_result
BOCA_test_result
COVS_test_result
ENTERO_test_result
BAC_test_result


## Inconsistencies

In [21]:
df_combined.columns

Index(['lab_id', 'test_id', 'test_kit', 'patient_id', 'sample_id', 'state',
       'location', 'date_testing', 'epiweek', 'age', 'sex', 'FLUA_test_result',
       'Ct_FluA', 'FLUB_test_result', 'Ct_FluB', 'VSR_test_result', 'Ct_VSR',
       'SC2_test_result', 'Ct_geneE', 'Ct_geneN', 'Ct_geneS', 'Ct_ORF1ab',
       'Ct_RDRP', 'geneS_detection', 'META_test_result', 'RINO_test_result',
       'PARA_test_result', 'ADENO_test_result', 'BOCA_test_result',
       'COVS_test_result', 'ENTERO_test_result', 'BAC_test_result'],
      dtype='object')

Verificando se um mesmo test_id + test_kit apresentam mais de um resultado para um mesmo patógeno

In [22]:
agg_test_rules = {
    col+'_kit': (col, lambda x: ','.join(x.tolist()))
    for col in df_combined.columns
    if col.endswith('_result')
}

df_os_non_contraditory_test_results = (
    df_combined
    .groupby(['test_id', 'test_kit'])
    .agg(**agg_test_rules)
)

In [23]:
for col in df_os_non_contraditory_test_results.columns:
    if col.endswith('_kit'):
        print(col, df_os_non_contraditory_test_results[col].unique())

# Devem ser apenas ['NT' 'Neg' 'Pos']

FLUA_test_result_kit ['NT' 'Neg,Neg,Neg' 'Pos,Pos' 'Neg,Neg' 'Pos,Pos,Pos'
 'Neg,Neg,Neg,Neg,Neg' 'NT,NT' 'Pos,Pos,Pos,Pos,Pos' 'Neg,Neg,Neg,Neg'
 'Neg']
FLUB_test_result_kit ['NT' 'Neg,Neg,Neg' 'Neg,Neg' 'Pos,Pos' 'Neg,Neg,Neg,Neg,Neg' 'NT,NT'
 'Neg' 'Neg,Neg,Neg,Neg' 'Pos,Pos,Pos' 'Pos,Pos,Pos,Pos,Pos' 'Pos'
 'NT,NT,NT']
VSR_test_result_kit ['NT' 'Neg,Neg,Neg' 'NT,NT' 'Pos,Pos,Pos' 'Neg,Neg,Neg,Neg,Neg'
 'Pos,Pos,Pos,Pos,Pos' 'Neg,Neg,Neg,Neg' 'Neg' 'Pos' 'Pos,Pos' 'NT,NT,NT']
SC2_test_result_kit ['Neg' 'Pos' 'NT,NT,NT' 'NT,NT' 'Pos,Pos,Pos,Pos,Pos' 'Pos,Pos'
 'Neg,Neg,Neg,Neg,Neg' 'NT' 'Neg,Neg' 'NT,NT,NT,NT' 'Neg,Neg,Neg,Neg'
 'Neg,Neg,Neg']
META_test_result_kit ['NT' 'NT,NT,NT' 'NT,NT' 'NT,NT,NT,NT,NT' 'NT,NT,NT,NT']
RINO_test_result_kit ['NT' 'NT,NT,NT' 'NT,NT' 'NT,NT,NT,NT,NT' 'NT,NT,NT,NT']
PARA_test_result_kit ['NT' 'NT,NT,NT' 'NT,NT' 'NT,NT,NT,NT,NT' 'NT,NT,NT,NT']
ADENO_test_result_kit ['NT' 'NT,NT,NT' 'NT,NT' 'NT,NT,NT,NT,NT' 'NT,NT,NT,NT']
BOCA_test_result_kit ['NT' 'NT,NT

Verificando se todos os Ids originais etão presentes no arquivo final

In [24]:
set_test_ids_final_df = set(df_combined['test_id'])
set_test_ids_raw_data = set(df_paineis['ACCESSION'].astype(int))

In [25]:
# Testar se todos os test_ids da planilha original estão na planilha final
# PODE RETORNAR UM SET NÃO VAZIO DADO QUE OS DADOS DE DENGUE FORAM REMOVIDOS
set_test_ids_raw_data.difference(set_test_ids_final_df)

{2022297010179,
 2023122010119,
 2022197002248,
 2022197002252,
 2022090014733,
 2022133006356,
 2023182008341,
 2023186006040,
 2022129008666,
 2023124008987,
 2022108004379,
 2023134003227,
 2023132004382,
 2023153008670,
 2023087013920,
 2022326009901,
 2023122010157,
 2023072006192,
 2022357008433,
 2023006011453,
 2022320013376,
 2023087013955,
 2022330007620,
 2022206013508,
 2023134003269,
 2023097008205,
 2022042009678,
 2023167000655,
 2022150013007,
 2022266011729,
 2023097008209,
 2022073008211,
 2022127009873,
 2023033012309,
 2023031013458,
 2023159006626,
 2022351011929,
 2022133006425,
 2022106005594,
 2023178010717,
 2022150013022,
 2022032015455,
 2023097008223,
 2022365003873,
 2022206013542,
 2022085001322,
 2023136002156,
 2023093010541,
 2023199015024,
 2022181011569,
 2022355009650,
 2022179012722,
 2023037010033,
 2022146015346,
 2023134003318,
 2022069010551,
 2022363005047,
 2022264012921,
 2023058014330,
 2022162006137,
 2022146015356,
 2023184007296,
 2023087

In [26]:
# Verificar se há test_ids na planilha final que não estão na planilha original
# deve retornar um set vazio
set_test_ids_final_df.difference(set_test_ids_raw_data)

set()

Verificando se todos os patógenos de um test_kit estão sendo testados para todos os test_id

In [27]:
PATHOGENS_TESTS = {
    'panel_21':{
        # PAINCOVI
        'FLUA_test_result',
        'FLUB_test_result',
        'VSR_test_result',
        'SC2_test_result',
        'META_test_result',
        'RINO_test_result',
        'PARA_test_result',
        'ADENO_test_result',
        'COVS_test_result',
        'BAC_test_result',
    },

    'panel_24':{
        # RESPIRA
        'FLUA_test_result',
        'FLUB_test_result',
        'VSR_test_result',
        'META_test_result',
        'RINO_test_result',
        'PARA_test_result',
        'ADENO_test_result',
        'BOCA_test_result',
        'COVS_test_result',
        'ENTERO_test_result',
        'BAC_test_result',
    },

    'panel_4':{
        # PCRESPSL & PCRVRESP
        'FLUA_test_result',
        'FLUB_test_result',
        'VSR_test_result',
        'SC2_test_result',
    },

    'covid_antigen':{
        'SC2_test_result',
    },

    'covid_pcr':{
        'SC2_test_result',
    },
}

In [28]:
# Mapeando Pos e Neg para Tes
df_combined_test_pathogen_non_nt_on_kit = (
    df_combined
    .replace(('Pos', 'Neg'),  'Tes')
)

In [29]:
for pathogen, test_columns in PATHOGENS_TESTS.items():
    print(pathogen)
    print(
        df_combined_test_pathogen_non_nt_on_kit
        .query("test_kit == @pathogen")
        [list(test_columns)]
        .drop_duplicates().T
        # Deve resultar em apenas uma linha completa por 'Tes'
    )

    print('\n\n')

panel_21
Empty DataFrame
Columns: []
Index: [FLUA_test_result, RINO_test_result, META_test_result, SC2_test_result, BAC_test_result, COVS_test_result, FLUB_test_result, PARA_test_result, ADENO_test_result, VSR_test_result]



panel_24
Empty DataFrame
Columns: []
Index: [FLUA_test_result, RINO_test_result, META_test_result, ENTERO_test_result, BAC_test_result, COVS_test_result, FLUB_test_result, PARA_test_result, ADENO_test_result, BOCA_test_result, VSR_test_result]



panel_4
Empty DataFrame
Columns: []
Index: [FLUA_test_result, VSR_test_result, FLUB_test_result, SC2_test_result]



covid_antigen
                  10
SC2_test_result  Tes



covid_pcr
                   0
SC2_test_result  Tes





Verificando resultados de alguns testes

In [30]:
import numpy as np

In [31]:
np.random.seed(214)

ids = np.random.choice(df_combined['test_id'], 100)
test_columns = [col for col in df_combined.columns if col.endswith('_result')]

In [32]:
current_id=50

In [33]:
current_id = current_id+1
id = ids[current_id]
id = 2023138006519
print( df_combined.query("test_id == @id")[['test_id', 'test_kit'] + test_columns].T )
print( df_paineis.query("ACCESSION == @id") [['ACCESSION', 'EXAME', 'DETALHE_EXAME', 'RESULTADO']].T )

                           630575         630576         630577  \
test_id             2023138006519  2023138006519  2023138006519   
test_kit                   test_4         test_4         test_4   
FLUA_test_result              Neg            Neg            Neg   
FLUB_test_result              Neg            Neg            Neg   
VSR_test_result               Neg            Neg            Neg   
SC2_test_result               Neg            Neg            Neg   
META_test_result               NT             NT             NT   
RINO_test_result               NT             NT             NT   
PARA_test_result               NT             NT             NT   
ADENO_test_result              NT             NT             NT   
BOCA_test_result               NT             NT             NT   
COVS_test_result               NT             NT             NT   
ENTERO_test_result             NT             NT             NT   
BAC_test_result                NT             NT             N

Verificando resultados de alguns testes - Fltrando por test_kit

In [34]:
np.random.seed(214)

ids = np.random.choice(df_combined.query("test_kit not in ('covid_pcr', 'covid_antigen')")['test_id'], 100)
test_columns = [col for col in df_combined.columns if col.endswith('_result')]

In [35]:
current_id=0

In [36]:
current_id = current_id+1
id = ids[current_id]
print( df_combined.query("test_id == @id")[['test_id', 'test_kit'] + test_columns].T )
print( df_paineis.query("ACCESSION == @id") [['ACCESSION', 'EXAME', 'DETALHE_EXAME', 'RESULTADO']].T )

                            59886          59887
test_id             2022011004230  2022011004230
test_kit                   test_2         test_2
FLUA_test_result              Neg            Neg
FLUB_test_result              Neg            Neg
VSR_test_result                NT             NT
SC2_test_result                NT             NT
META_test_result               NT             NT
RINO_test_result               NT             NT
PARA_test_result               NT             NT
ADENO_test_result              NT             NT
BOCA_test_result               NT             NT
COVS_test_result               NT             NT
ENTERO_test_result             NT             NT
BAC_test_result                NT             NT
                                             661131  \
ACCESSION                             2022011004230   
EXAME          PESQUISA RÁPIDA PARA INFLUENZA A E B   
DETALHE_EXAME             INFLUENZA A, TESTE RÁPIDO   
RESULTADO                             NÃO DET